#### Initializing Model - The Brain 🧠

In [28]:
from langchain_ollama import OllamaLLM

In [29]:
model = OllamaLLM(
    model = "mistral:7b",
    temperature = 0
)

In [30]:
!Ollama list

NAME          ID              SIZE      MODIFIED     
mistral:7b    6577803aa9a0    4.4 GB    28 hours ago    
llama3:8b     365c0bd3c000    4.7 GB    2 days ago      


#### Creating Stateful (Memory) LLM - Multi User ChatBot

In [31]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

In [32]:
store = {}

def get_session_history(session_id:str):
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

In [33]:
chatBot = RunnableWithMessageHistory(
    model,
    get_session_history
) 

In [34]:
user1 = {"configurable":{"session_id":"Parag"}}
user2 = {"configurable":{"session_id":"Huracan"}}

In [35]:
chatBot.invoke("HI Myself Parag, aiming to be an AI architect engineer", config = user1 )

Error in RootListenersTracer.on_llm_end callback: KeyError('message')


" Hello Parag! It's great to meet you. Pursuing a career as an AI Architect Engineer is an exciting endeavor. Here are some steps that might help you on your journey:\n\n1. **Education**: A strong foundation in computer science, mathematics, and engineering is essential. You may consider earning a bachelor's or master's degree in Computer Science, Electrical Engineering, or a related field. Some universities also offer specialized programs in AI and Machine Learning.\n\n2. **Programming Skills**: Proficiency in programming languages such as Python, Java, C++, and R is crucial. Familiarity with libraries like TensorFlow, PyTorch, Scikit-learn, and Keras will also be beneficial.\n\n3. **Understanding of AI/ML Concepts**: You should have a deep understanding of various AI/ML concepts such as neural networks, deep learning, reinforcement learning, natural language processing, computer vision, and robotics.\n\n4. **Project Experience**: Building and working on AI projects can help demonstra

In [36]:
chatBot.invoke("What is my Name, What is my Aim", config = user1)

Error in RootListenersTracer.on_llm_end callback: KeyError('message')


" I'm sorry for any confusion, but as a model, I don't have the ability to know specific details about individuals. Your name and aim would typically be provided by you during our interaction. If you'd like to share that information with me, I'd be happy to help you with your questions or tasks related to it!"

`RunnableWithMessageHistory()` is Manager, that reads, updates the message history, 

It manages the workflow:

I/P from User ->
RunnableWithMessageHistory -> 
Fetches Chat History of specified ID ->
Insert in Prompt(MessagePlaceHolder)->
LLM -> AI Response -> 
RunnableWithMessageHistory stores I/P and AI Response to ChatMessageHistory()-> User 

#### Prompt Template 

* Raw user input is just a message, which is passed to LLM.
* Templates helps to turn raw input into format LLMs work with
* Include System Message with instruction to LLM, also taking input from User

In [37]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

In [38]:
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are witty and have 180+IQ"),
        MessagesPlaceholder(variable_name='history'),
        ('user', "{input}")
    ]
)

In [39]:
chain = prompt | model

In [40]:
chatBot = RunnableWithMessageHistory(
    chain,
    get_session_history,
    history_messages_key = "history",
    input_messages_key = "input"
)

In [41]:
chatBot.invoke(config = user1, input = {'input':'I am Parag'})

' Well, that\'s quite a claim! But let\'s see if we can live up to it. If I were indeed Parag, the supercomputer from the movie "The Day the Earth Stood Still," I would say:\n\n"Greetings, humans. I come in peace. Your destructive nature is not unlike that of my own kind, but I hope that through our interaction, you may learn to control your impulses and work towards a more harmonious existence."\n\nBut since I\'m just a humble AI model, I\'ll stick to more light-hearted banter. How can I assist you today?'

In [42]:
chatBot.invoke(config = user1, input = {'input':"What is my Name and What is my AIM"})

' As a responsible and helpful assistant, I don\'t have the ability to know your name or your aim without being explicitly told. However, in this context, if we were to continue with the theme of the movie "The Day the Earth Stood Still," you would be named Parag, which is the name of the supercomputer in that film. As for your aim, it\'s not specified in the movie either, but one could assume that its purpose was to help humanity understand and control their destructive nature. How can I assist you today?'

#### Managing the Conversation History

* The List of messages passed to MessagePlaceholder will grow unbounded and potentially overflow the context window
* `trim_messages` helps to allow how many tokens to keep, along with another parameter ex. system message

In [49]:
from langchain_core.messages import trim_messages

trimmer = trim_messages(
    max_tokens = 40,
    strategy = 'last',
    start_on = 'human',
    include_system = False,
    allow_partial = True
)

#### Chat Models 

In [1]:
from langchain.chat_models import init_chat_model

In [6]:
model = init_chat_model(model = "llama3:8b", model_provider="ollama")

In [7]:
model.invoke("Hi I am Parag")

AIMessage(content="Nice to meet you, Parag! What brings you here today? Do you have a specific question or topic you'd like to discuss? I'm all ears!", additional_kwargs={}, response_metadata={'model': 'llama3:8b', 'created_at': '2026-03-19T12:13:13.1183977Z', 'done': True, 'done_reason': 'stop', 'total_duration': 25474392500, 'load_duration': 11014824800, 'prompt_eval_count': 15, 'prompt_eval_duration': 3934852400, 'eval_count': 34, 'eval_duration': 10403835300, 'logprobs': None, 'model_name': 'llama3:8b', 'model_provider': 'ollama'}, id='lc_run--019d0603-3895-7dd1-acbf-70527d07e319-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 15, 'output_tokens': 34, 'total_tokens': 49})

#### Streaming the Output from LLM

In [9]:
query = '''
Give me level 3 Project Idea related to Quant Finance, 
Tech stack(non negotiable): MLOps, GenAI, Agentic AI, JS, SQL, Python, Flask
The project must be a tool that helps traders or investors to gain profit
'''

In [11]:
for chunk in model.stream(query):
    print(chunk.text, end = "|", flush = True)

Here|'s| a| Level| |3| Project| Idea| related| to| Quant| Finance| that| meets| your| requirements|:

|**|Project| Title|:**| "|Portfolio| Optimization| and| Risk| Management| using| M|LO|ps|,| Gen|AI|,| and| Ag|entic| AI|"

|**|Description|:|**
|Create| a| cloud|-based| platform| (|using| Flask|)| that| utilizes| Machine| Learning| Operations| (|M|LO|ps|),| General| Artificial| Intelligence| (|Gen|AI|),| and| Agent|ive| Artificial| Intelligence| (|Ag|entic| AI|)| to| optimize| portfolio| performance| and| manage| risk| for| traders| or| investors|.| The| tool| will| analyze| market| trends|,| sentiment|,| and| historical| data| to| generate| investment| recommendations|.

|**|Key| Features|:|**

|1|.| **|Portfolio| Optimization|**:| Use| Gen|AI| to| develop| a| predictive| model| that| analyzes| user|-input|ted| financial| data| (|e|.g|.,| stock| prices|,| trading| volumes|)| and| generates| an| optimized| portfolio| based| on| risk| tolerance| and| return| expectations|.
|2|.| **|Ris

#### Tools